# Energy Forecast — Prediction Performance (last 30 days)

Pulls live data from Home Assistant via SMB, fetches weather from Open-Meteo, and renders accuracy charts for the last 30 days.

**Pre-requisite:** `SMB_PASSWORD` environment variable must be set before starting the kernel.


In [ ]:
from __future__ import annotations

import io
import json
import os
from datetime import date, timedelta

import altair as alt
import pandas as pd
import requests
import yaml
from smb.SMBConnection import SMBConnection


In [ ]:
# ── Credentials ──────────────────────────────────────────────────────────────
SMB_USER = os.getenv("SMB_USER", "martin")
SMB_PASSWORD = os.getenv("SMB_PASSWORD")
if not SMB_PASSWORD:
    raise RuntimeError(
        "SMB_PASSWORD environment variable is not set. "
        "Set it before starting the kernel: export SMB_PASSWORD=<password>"
    )

# ── Constants ─────────────────────────────────────────────────────────────────
HA_HOST = "homeassistant"
SMB_SHARE = "addon_configs"
AD_BASE = "a0d7b954_appdaemon/apps"
FORECAST_REMOTE = f"{AD_BASE}/energy_forecast"

TZ = "Europe/Zurich"
CUTOFF_DAYS = 30
EV_THRESHOLD_KWH = 7.0  # matches EV_CHARGING_THRESHOLD_KWH in const.py

WEEKDAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

print("Config OK — SMB_PASSWORD is set")